# InfiniTune Colab v2: Real-Time LLM Fine-Tuning Framework

This notebook runs the **real InfiniTune streaming architecture** on Google Colab GPUs (T4, L4, or A100).
Unlike legacy notebooks that simulated iterators without streaming infrastructure, v2 runs the repository's native `producer.py`, `trainer.py`, `inference.py`, Apache Kafka 3.9 in KRaft mode, and the reproducible `benchmarks.run` harness.

### Cell 1: Environment & GPU Hardware Check
Inspect the allocated NVIDIA GPU, compute capability, and establish the precision policy (`bf16` on Ampere/Ada/Hopper vs `fp16` on Turing T4).

In [ ]:
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU available. In Colab, go to Runtime -> Change runtime type -> T4 / L4 / A100 GPU.")

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"Active GPU: {device_name}")
print(f"Compute Capability: {capability[0]}.{capability[1]}")

# Ampere+ (SM 8.0+) natively supports bf16; Turing (T4, SM 7.5) uses fp16
recommended_precision = "bf16" if capability[0] >= 8 else "fp16"
print(f"Recommended precision policy: {recommended_precision}")

### Cell 2: Mount Google Drive for Persistent Storage
Mount Google Drive under `/content/drive` to persist checkpoints, benchmark results, Hugging Face model caches, and logs across session disconnects.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/infinitune")
PERSIST.mkdir(parents=True, exist_ok=True)
print(f"Persistent storage directory: {PERSIST}")

### Cell 3: Clone Repository at Pinned Commit
Clone the official InfiniTune repository and checkout the verified evaluation commit.

In [ ]:
import subprocess
import os

REPO_URL = "https://github.com/rohan-sawale/Infinitune-Realtime-LLM-Fine-Tuning-Framework.git"
REPO_DIR = "/content/Infinitune-Realtime-LLM-Fine-Tuning-Framework"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
print(f"InfiniTune Repository checked out at commit: {commit_hash}")

### Cell 4: Install Colab Dependencies & Run Environment Census
Install dependencies from `requirements-colab.txt` (preserving Colab's preinstalled CUDA-enabled PyTorch, NumPy, and scikit-learn), followed by running `benchmarks/env_check.py`.

In [ ]:
!pip install -r requirements-colab.txt
!python -m benchmarks.env_check

### Cell 5: Start Apache Kafka (KRaft mode)
Execute `scripts/colab/start_kafka.sh` to install headless JRE, download Kafka 3.9.0, format KRaft storage under `/content/kafka-data`, start the broker, and verify readiness on `localhost:9092`.

In [ ]:
!bash scripts/colab/start_kafka.sh

### Cell 6: Configure Hugging Face Cache & Optional Secrets
Set `HF_HOME` to persistent Drive storage so base model weights are downloaded only once. Safely retrieve optional Hugging Face tokens from Colab Secrets.

In [ ]:
import os
os.environ["HF_HOME"] = str(PERSIST / "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print(f"Hugging Face Cache: {os.environ['HF_HOME']}")

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN successfully loaded from Colab Secrets.")
except Exception:
    print("HF_TOKEN secret not set (public model downloads will still work).")

### Cell 7: Run End-to-End Benchmark Suite
Execute a reproducible benchmark suite via `benchmarks.run`. Outputs and manifests will be written directly to persistent storage on Drive.

In [ ]:
import time

RUN_NAME = f"colab_run_{int(time.time())}"
RUN_DIR = PERSIST / "runs" / RUN_NAME

# Example: Run streaming_quality suite with GPU acceleration
!python -m benchmarks.run run \
    --suites streaming_quality \
    --profile standard \
    --model distilgpt2 \
    --live-config configs/imdb_quantitative.yaml \
    --device cuda \
    --kafka-bootstrap localhost:9092 \
    --run-dir {RUN_DIR}

### Cell 8: Resume Interrupted Benchmark Run
If the session is preempted or interrupted, resume execution using the identical `--run-dir`.

In [ ]:
# To resume an existing run after reconnect:
!python -m benchmarks.run run \
    --suites streaming_quality \
    --profile standard \
    --model distilgpt2 \
    --live-config configs/imdb_quantitative.yaml \
    --device cuda \
    --kafka-bootstrap localhost:9092 \
    --run-dir {RUN_DIR}

### Cell 9: Archive & Export Benchmark Results
Create a compressed zip archive of the benchmark run directory for export and documentation.

In [ ]:
import shutil

zip_target = PERSIST / f"{RUN_NAME}.zip"
shutil.make_archive(str(zip_target.with_suffix("")), "zip", RUN_DIR)
print(f"Archive written to: {zip_target}")